# Risk labels on our forecasts

Applies the risk flags of [`risk-definition.ipynb`](risk-definition.ipynb) (thresholds, day rules, limits of the label) to the test-window forecasts of spec 09's picks ([`regression-visualization-best-models.ipynb`](../05_modeling/regression-visualization-best-models.ipynb)) and of SMARD, and scores them against the actual labels; forecast method and caveats are in [`regression-models-claude.ipynb`](../05_modeling/regression-models-claude.ipynb).
Reads `data/risk_classification/risk_labels_{daily,hourly}.csv`, `data/models/model_{forecast_errors_hourly,scoreboard}.csv` and `data/metrics/smard_forecast_errors_hourly.csv`; writes `data/risk_classification/model_risk_labels_{daily,hourly}.csv`.
*The label is a national-balance proxy, not a validated intervention record.*

## 1 Setup and checks

### 1.1 Settings

In [ ]:
from pathlib import Path

import holidays
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle

PICKS = {"high": "xgb_hybrid", "low": "random_forest_hybrid"}  # spec 09 rank 1: high_extreme / low_extreme
PICK_CATEGORIES = {"high": ["high_extreme"], "low": ["low_extreme", "below_zero"]}  # checked in 1.3
SPLIT = "rolling"            # split method of the picks (spec 09's CANDIDATE_SPLIT)
BASES = {"high": ["rolling"], "low": ["rolling", "zero"]}
RULES = ["any", "3h"]
MIN_BIN_SHARE = 0.5          # spec 09's forecast-share floor, used by the pick check
ZOOM_WEEKS = {"high": [None, None], "low": [None, None]}  # a Monday "YYYY-MM-DD" overrides a zoom rule
EXPORT_ENABLED = True
MARGIN_WINDOW = 10_000       # MWh: axis range of the day-margin plot around 0

# Day rules as durations, as in risk-definition.ipynb §3.1 (RESOLUTION is measured in §2.1)
PERSISTENCE = pd.Timedelta("3h")
DAY_COMPLETENESS = 23 / 24

# Label and colour per row, copied from regression-visualization-best-models.ipynb
STYLE = {
    "sarimax_fourier": {"label": "SARIMAX + Fourier", "color": "#D9A53A"},
    "lgbm_direct": {"label": "LightGBM direct", "color": "#2C6EBA"},
    "lgbm_hybrid": {"label": "LightGBM hybrid", "color": "#2F8F5B"},
    "xgb_direct": {"label": "XGBoost direct", "color": "#E95D0F"},
    "xgb_hybrid": {"label": "XGBoost hybrid", "color": "#B10F0F"},
    "linear_direct": {"label": "Ridge direct", "color": "#7A4FA3"},
    "random_forest_hybrid": {"label": "Random forest hybrid", "color": "#8C564B"},
    "actual": {"label": "Actual residual load", "color": "#1C1C1C"},
    "smard": {"label": "SMARD day-ahead", "color": "#48505A", "linestyle": "--"},
}

# Thresholds, flagged hours and hit cells per (direction, basis), never a model's colour
TAIL_COLOR = {
    ("high", "rolling"): "#E6B800",  # gold, spec 09's high_extreme
    ("low", "rolling"): "#17BECF",   # cyan, low_extreme
    ("low", "zero"): "#9EDAE5",      # light cyan, below_zero
}

# Day and hour outcomes; a hit takes the TAIL_COLOR, flagged outcomes are edged in it
OUTCOME_COLOR = {
    "miss": "#1C1C1C",
    "false alarm": "#AEB8C5",
    "quiet": "#F2F4F7",
    "not evaluable": "white",  # also days outside the test window
}


def style_timeseries(ax, title, ylabel):
    """Apply the EDA time-series chart style to the matplotlib axes.

    `ylabel` is required: every plot must state its unit.
    """
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel, color="grey")

    # Light horizontal grid lines behind the data, no box on the top and right
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

    # Tick labels without tick marks
    ax.tick_params(colors="black", length=0)

    # x axis: one label per year, minor ticks every quarter
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_minor_locator(mdates.MonthLocator((1, 4, 7, 10)))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

    # y axis: thousands separator, no decimals (e.g. 45,000)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")


def style_plain(ax, title, ylabel, xlabel):
    """The same look for plots without a time axis."""
    ax.set_title(title, loc="center", fontsize=15, pad=12)
    ax.set_xlabel(xlabel, color="grey")
    ax.set_ylabel(ylabel, color="grey")
    ax.grid(axis="y", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.tick_params(colors="black", length=0)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")

### 1.2 Loading

In [ ]:
# data/: first in the working directory, then in each parent
DATA_DIR = next(
    (folder / "data" for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "data").is_dir()),
    None,
)
if DATA_DIR is None:
    raise RuntimeError(f"No data/ folder found in {Path.cwd()} or any parent folder.")

LABELS_SOURCE = "notebooks/03_risk_classification/risk-definition.ipynb"
MODEL_SOURCE = "notebooks/05_modeling/regression-models-claude.ipynb with EXPORT_ENABLED = True"
SMARD_SOURCE = "notebooks/02_forecast_metrics/forecast-metrics-claude.ipynb"
SOURCES = {
    "labels_daily": ("risk_classification/risk_labels_daily.csv", LABELS_SOURCE),
    "labels_hourly": ("risk_classification/risk_labels_hourly.csv", LABELS_SOURCE),
    "hourly": ("models/model_forecast_errors_hourly.csv", MODEL_SOURCE),
    "scoreboard": ("models/model_scoreboard.csv", MODEL_SOURCE),
    "smard": ("metrics/smard_forecast_errors_hourly.csv", SMARD_SOURCE),
}
STAMP = "%Y-%m-%d %H:%M:%S"


def load(name):
    """One file as a DataFrame; stops if it is missing."""
    path, producer = SOURCES[name]
    path = DATA_DIR / path
    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Create it by running {producer}.")
    frame = pd.read_csv(path)
    if "timestamp" in frame.columns:
        frame["timestamp"] = pd.to_datetime(frame["timestamp"], format=STAMP)
    if "date" in frame.columns:
        frame["date"] = pd.to_datetime(frame["date"], format="%Y-%m-%d")
    return frame


# Label columns stay three-state: an empty cell is "not evaluable", never False
LABEL_PAIRS = [(direction, basis) for direction in BASES for basis in BASES[direction]]
DAILY_THRESHOLDS = [f"{d}_threshold_{b}" for d, b in LABEL_PAIRS]
DAILY_FLAGS = [f"{d}_risk_{b}_{r}" for d, b in LABEL_PAIRS for r in RULES]
DAILY_RANGES = [f"{d}_range_{e}_{b}_{r}" for d, b in LABEL_PAIRS for r in RULES for e in ("start", "end")]
HOURLY_FLAGS = [f"{d}_risk_hour_{b}" for d, b in LABEL_PAIRS]

labels_daily = load("labels_daily").set_index("date")[DAILY_THRESHOLDS + DAILY_FLAGS + DAILY_RANGES]
labels_daily[DAILY_FLAGS] = labels_daily[DAILY_FLAGS].astype("boolean")
for column in DAILY_RANGES:
    labels_daily[column] = pd.to_datetime(labels_daily[column], format=STAMP)
labels_hourly = load("labels_hourly").set_index("timestamp")[["residual_load"] + HOURLY_FLAGS]
labels_hourly[HOURLY_FLAGS] = labels_hourly[HOURLY_FLAGS].astype("boolean")

hourly = load("hourly")
scoreboard = load("scoreboard")
smard = load("smard").set_index("timestamp")

# Test window: the hours and local dates of the model export
TEST_HOURS = pd.DatetimeIndex(sorted(hourly["timestamp"].unique()))
TEST_DAYS = pd.DatetimeIndex(sorted(TEST_HOURS.normalize().unique()))

# The picks' forecasts; a missing pick stays an empty column and fails the pick check
picked = hourly[hourly["split_method"].eq(SPLIT) & hourly["model"].isin(PICKS.values())]
FORECAST = picked.pivot(index="timestamp", columns="model", values="forecast").reindex(
    index=TEST_HOURS, columns=list(dict.fromkeys(PICKS.values()))
)
ACTUAL = picked.groupby("timestamp")["residual_load"].first().reindex(TEST_HOURS)
SMARD_FORECAST = smard["fc_residual_load"].reindex(TEST_HOURS)

# Common hours: an actual, SMARD's forecast and a forecast from both picks
COMMON = TEST_HOURS[ACTUAL.notna() & SMARD_FORECAST.notna() & FORECAST.notna().all(axis=1)]

print(f"test window : {TEST_DAYS[0]:%Y-%m-%d} .. {TEST_DAYS[-1]:%Y-%m-%d} ({len(TEST_DAYS)} days, {len(TEST_HOURS):,} hours)")
print(f"picks       : high {PICKS['high']}, low {PICKS['low']} ({SPLIT} split)")
print(f"common hours: {len(COMMON):,} of {len(TEST_HOURS):,}")

### 1.3 Pick check

Each pick needs a `SPLIT` row in both model exports and must pass spec 09's bin rule in every `PICK_CATEGORIES` category. On a failure the notebook stops; after spec 09 is re-run with new picks, edit `PICKS`.

In [ ]:
SMARD_ROW = ("smard", "none")
VALUE = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="value")
COUNT = scoreboard.pivot(index=["model", "split_method"], columns="metric", values="count")
HOURLY_ROWS = set(zip(hourly["model"], hourly["split_method"]))


def failed_rules(row, category):
    """Spec 09's bin rule: the rules a row fails in a category; empty if it qualifies."""
    failed = []
    for view in ["actual", "forecast"]:
        metric = f"MAE_{category}_by_{view}"
        if not VALUE.loc[row, metric] < VALUE.loc[SMARD_ROW, metric]:
            failed.append(f"{metric} {VALUE.loc[row, metric]:,.0f} not below SMARD's {VALUE.loc[SMARD_ROW, metric]:,.0f}")
    by_actual = COUNT.loc[row, f"MAE_{category}_by_actual"]
    share = COUNT.loc[row, f"MAE_{category}_by_forecast"] / by_actual if by_actual else np.nan
    if not share >= MIN_BIN_SHARE:
        failed.append(f"forecast share {share:.2f} < {MIN_BIN_SHARE}")
    return failed


problems = []
for direction, model in PICKS.items():
    row = (model, SPLIT)
    if row not in HOURLY_ROWS:
        problems.append(f"{direction} {model}: no {SPLIT} row in model_forecast_errors_hourly.csv")
    if row not in VALUE.index:
        problems.append(f"{direction} {model}: no {SPLIT} row in model_scoreboard.csv")
        continue
    for category in PICK_CATEGORIES[direction]:
        failed = failed_rules(row, category)
        problems += [f"{direction} {model} in {category}: {rule}" for rule in failed]
        by_actual, by_forecast = (f"MAE_{category}_by_{view}" for view in ["actual", "forecast"])
        print(
            f"{direction:<4} {model:<20} {category:<12} "
            f"MAE by actual {VALUE.loc[row, by_actual]:,.0f} (SMARD {VALUE.loc[SMARD_ROW, by_actual]:,.0f}), "
            f"by forecast {VALUE.loc[row, by_forecast]:,.0f} (SMARD {VALUE.loc[SMARD_ROW, by_forecast]:,.0f}), "
            f"forecast share {COUNT.loc[row, by_forecast] / COUNT.loc[row, by_actual]:.2f}: "
            f"{'fails' if failed else 'passes'}"
        )

if problems:
    raise RuntimeError(
        f"pick check failed for PICKS = {PICKS}:\n  " + "\n  ".join(problems)
        + "\nRe-run regression-visualization-best-models.ipynb and edit PICKS; no model is picked here."
    )
print("pick check passed")

### 1.4 Snapshot self-check

Labels, model exports and SMARD file must come from one `smard.csv` fetch; the notebook never joins on a partial overlap.

In [ ]:
REGENERATE = "API-connection → risk-definition → forecast-metrics-claude → regression-models-claude"


def require(ok, problem):
    """Stop the notebook, naming the regeneration order."""
    if not ok:
        raise RuntimeError(f"self-check failed: {problem}. Regenerate in this order: {REGENERATE}.")


require(len(COMMON) > 0, "no common hours")

# Every test day and hour has a label row
missing_days = TEST_DAYS.difference(labels_daily.index)
require(missing_days.empty, f"{len(missing_days)} test days missing from risk_labels_daily.csv")
missing_hours = TEST_HOURS.difference(labels_hourly.index)
require(missing_hours.empty, f"{len(missing_hours)} test hours missing from risk_labels_hourly.csv")

# Same actuals in the three hourly files
for name, other in [
    ("smard_forecast_errors_hourly.csv", smard["residual_load"]),
    ("risk_labels_hourly.csv", labels_hourly["residual_load"]),
]:
    gap = float((ACTUAL[COMMON] - other.reindex(COMMON)).abs().max())
    require(gap <= 1e-6, f"residual_load of the model export and {name} differ by up to {gap:,.2f} MWh")

print(
    f"self-check passed: labels, model exports and SMARD file share one snapshot "
    f"({len(TEST_DAYS)} days, {len(COMMON):,} common hours)"
)

## 2 Labels

### 2.1 Rule functions

Copied from `risk-definition.ipynb` §3.1, §3.2 and §5.1. One change: `day_rules` counts a day only when it is evaluable (2.2), in place of `day_complete`.

In [ ]:
# Resolution is measured, not assumed.
RESOLUTION = TEST_HOURS.to_series().diff().mode().iloc[0]
MIN_RUN = int(PERSISTENCE / RESOLUTION)
EXPECTED_OBS_PER_DAY = int(pd.Timedelta("1D") / RESOLUTION)
MIN_OBS_PER_DAY = int(np.ceil(DAY_COMPLETENESS * EXPECTED_OBS_PER_DAY))

# The rules run on the common hours of the test days
DAY = pd.Series(COMMON.normalize(), index=COMMON)
DAYS = TEST_DAYS


def hourly_crossings(series, threshold_by_day, direction):
    """Per-observation crossing flags against each observation's own day's threshold.

    Returns nullable booleans: pd.NA wherever the day's threshold is undefined, so an undefined
    threshold can never masquerade as "not at risk".
    """
    thr = pd.Series(
        threshold_by_day.reindex(series.index.normalize()).to_numpy(), index=series.index
    )
    crossed = (series >= thr) if direction == "high" else (series <= thr)
    return crossed.astype("boolean").mask(thr.isna())


def day_rules(hourly_flag, threshold_by_day):
    """The `any` and `3h` day rules from per-observation crossing flags.

    `any`  — at least one observation crosses.
    `3h`   — the crossing persists for at least PERSISTENCE, consecutively, within the day.

    Consecutive *rows* count as consecutive: on the spring-DST day the missing 02:00 makes
    a run spanning it one hour longer in wall-clock terms than in rows. The approximation is noted
    rather than corrected.

    Days with an undefined threshold, or not evaluable, are pd.NA in both rules rather than False.
    """
    filled = hourly_flag.fillna(False).astype(bool)
    any_rule = filled.groupby(DAY).any().reindex(DAYS, fill_value=False)

    run_id = (filled != filled.shift()).cumsum()
    run_len = filled.groupby([DAY, run_id]).transform("size").where(filled, 0)
    persist_rule = (run_len.groupby(DAY).max() >= MIN_RUN).reindex(DAYS, fill_value=False)

    valid = (threshold_by_day.notna() & EVALUABLE).reindex(DAYS, fill_value=False)
    return any_rule.astype("boolean").mask(~valid), persist_rule.astype("boolean").mask(~valid)


def flagged_range(hourly_flag, rule):
    """Earliest and latest flagged observation per day, as a span (see §5.1).

    Under the `3h` rule only observations belonging to a run of at least PERSISTENCE count.
    """
    filled = hourly_flag.fillna(False).astype(bool)
    if rule == "3h":
        run_id = (filled != filled.shift()).cumsum()
        run_len = filled.groupby([DAY, run_id]).transform("size")
        filled = filled & (run_len >= MIN_RUN)

    stamps = pd.Series(filled.index[filled], index=filled.index[filled].normalize())
    grouped = stamps.groupby(level=0)
    return grouped.min().reindex(DAYS), grouped.max().reindex(DAYS)

### 2.2 Flags per source

The actual's flags and ranges are read from the label files; the model's and SMARD's come from the rule functions against the exported thresholds. A day is evaluable when its actual labels are set and its common hours cover `DAY_COMPLETENESS`; on any other day every flag is empty for every source. The two frames built here are the export layout of §6 and the only input of §3–§5.

In [ ]:
FLAG_SOURCES = ["actual", "model", "smard"]
THRESHOLD = {(d, b): labels_daily[f"{d}_threshold_{b}"].reindex(TEST_DAYS) for d, b in LABEL_PAIRS}

# Evaluable: actual labels set and enough common hours
common_per_day = pd.Series(1, index=COMMON).groupby(COMMON.normalize()).size().reindex(TEST_DAYS, fill_value=0)
EVALUABLE = labels_daily.loc[TEST_DAYS, DAILY_FLAGS].notna().all(axis=1) & common_per_day.ge(MIN_OBS_PER_DAY)
HOUR_EVALUABLE = EVALUABLE.reindex(COMMON.normalize()).to_numpy()

# The hourly series each source is flagged on
SERIES = {
    "actual": {"high": ACTUAL[COMMON], "low": ACTUAL[COMMON]},
    "model": {d: FORECAST.loc[COMMON, PICKS[d]] for d in BASES},
    "smard": {"high": SMARD_FORECAST[COMMON], "low": SMARD_FORECAST[COMMON]},
}


def rule_flags(series, direction, basis):
    """Hour flags, day flags and flagged ranges of a series against the exported threshold."""
    threshold = THRESHOLD[(direction, basis)]
    hour = hourly_crossings(series, threshold, direction).mask(~HOUR_EVALUABLE)
    any_rule, persist_rule = day_rules(hour, threshold)
    return hour, {"any": any_rule, "3h": persist_rule}, {rule: flagged_range(hour, rule) for rule in RULES}


def file_flags(direction, basis):
    """The same three, read from the label files."""
    hour = labels_hourly.loc[COMMON, f"{direction}_risk_hour_{basis}"].mask(~HOUR_EVALUABLE)
    days = {rule: labels_daily.loc[TEST_DAYS, f"{direction}_risk_{basis}_{rule}"].mask(~EVALUABLE) for rule in RULES}
    ranges = {
        rule: tuple(labels_daily.loc[TEST_DAYS, f"{direction}_range_{edge}_{basis}_{rule}"].where(EVALUABLE) for edge in ("start", "end"))
        for rule in RULES
    }
    return hour, days, ranges


day_columns = {
    "evaluable": EVALUABLE,
    "high_model": PICKS["high"],
    "low_model": PICKS["low"],
    **{f"{d}_threshold_{b}": THRESHOLD[(d, b)] for d, b in LABEL_PAIRS},
}
by_day = lambda series: series.groupby(series.index.normalize())
for source in FLAG_SOURCES:
    day_columns[f"{source}_max"] = by_day(SERIES[source]["high"]).max()
    day_columns[f"{source}_min"] = by_day(SERIES[source]["low"]).min()
hour_columns = {
    "residual_load": ACTUAL[COMMON],
    "high_model_forecast": SERIES["model"]["high"],
    "low_model_forecast": SERIES["model"]["low"],
    "fc_residual_load": SMARD_FORECAST[COMMON],
}
flag_columns, range_columns = {}, {}
for source in FLAG_SOURCES:
    for d, b in LABEL_PAIRS:
        hour, days, ranges = file_flags(d, b) if source == "actual" else rule_flags(SERIES[source][d], d, b)
        hour_columns[f"{source}_{d}_risk_hour_{b}"] = hour
        for rule in RULES:
            flag_columns[f"{source}_{d}_risk_{b}_{rule}"] = days[rule]
            for edge, values in zip(("start", "end"), ranges[rule]):
                range_columns[f"{source}_{d}_range_{edge}_{b}_{rule}"] = values

DAILY_EXPORT = pd.DataFrame(day_columns | flag_columns | range_columns, index=TEST_DAYS).rename_axis("date")
HOURLY_EXPORT = pd.DataFrame(hour_columns, index=COMMON).rename_axis("timestamp")

print(f"evaluable days: {EVALUABLE.sum()} of {len(TEST_DAYS)} test days ({(~EVALUABLE).sum()} excluded)")
counts = pd.DataFrame(
    {
        (source, column): [
            int(DAILY_EXPORT[f"{source}_{d}_risk_{b}_{rule}"].sum()) if column != "hours"
            else int(HOURLY_EXPORT[f"{source}_{d}_risk_hour_{b}"].sum())
            for d, b in LABEL_PAIRS
        ]
        for source in FLAG_SOURCES
        for rule, column in [("any", "any days"), ("3h", "3h days"), ("any", "hours")]
    },
    index=pd.MultiIndex.from_tuples(LABEL_PAIRS, names=["direction", "basis"]),
)
counts

### 2.3 Rule reproduction check

On every test day whose hours are all common, the rule functions applied to the actual must reproduce the label files exactly: day flags, ranges and hour flags. On a day that lost a common hour the file's flag may rest on an hour the forecasts don't have, so such days are counted, not checked.

In [ ]:
hours_per_day = pd.Series(1, index=TEST_HOURS).groupby(TEST_HOURS.normalize()).size()
FULLY_COMMON = TEST_DAYS[common_per_day.eq(hours_per_day).to_numpy()]
in_full = COMMON.normalize().isin(FULLY_COMMON)


def differs(a, b):
    """Where two series disagree; two empty cells agree."""
    one_empty = a.isna() != b.isna()
    return one_empty | (a.notna() & b.notna() & (a != b)).astype(bool)


problems = []
for d, b in LABEL_PAIRS:
    hour, days, ranges = rule_flags(ACTUAL[COMMON], d, b)
    pairs = {f"{d}_risk_hour_{b}": (hour[in_full], labels_hourly.loc[COMMON[in_full], f"{d}_risk_hour_{b}"])}
    for rule in RULES:
        pairs[f"{d}_risk_{b}_{rule}"] = (days[rule], labels_daily[f"{d}_risk_{b}_{rule}"])
        for edge, values in zip(("start", "end"), ranges[rule]):
            pairs[f"{d}_range_{edge}_{b}_{rule}"] = (values, labels_daily[f"{d}_range_{edge}_{b}_{rule}"])
    for column, (rebuilt, exported) in pairs.items():
        index = rebuilt.index if column.endswith(f"hour_{b}") else FULLY_COMMON
        wrong = int(differs(rebuilt.reindex(index), exported.reindex(index)).sum())
        if wrong:
            problems.append(f"{column}: {wrong} rows differ")

if problems:
    raise RuntimeError(
        "rule reproduction check failed; the copied rule functions no longer match risk-definition.ipynb:\n  "
        + "\n  ".join(problems)
    )
print(
    f"rule reproduction check passed on {len(FULLY_COMMON)} fully common test days "
    f"({len(TEST_DAYS) - len(FULLY_COMMON)} days lost a common hour and are not checked)"
)

### 2.4 Structural checks

On every source: `3h` implies `any`, a range exists exactly where the day is flagged, the day `any` flag equals "at least one flagged hour", and a flag is empty exactly where the day is not evaluable. §6 repeats them on the written files. Days flagged in both directions are reported, not prevented (`risk-definition.ipynb` §4.2); a non-zero count hints at a join error.

In [ ]:
def structural_problems(daily, hourly):
    """The Behaviour 9 checks on export-layout frames; empty if all pass."""
    problems = []
    evaluable = daily["evaluable"].astype(bool)
    for source in FLAG_SOURCES:
        for d, b in LABEL_PAIRS:
            flag = {rule: daily[f"{source}_{d}_risk_{b}_{rule}"] for rule in RULES}
            hour = hourly[f"{source}_{d}_risk_hour_{b}"]
            hour_any = hour.eq(True).groupby(hour.index.normalize()).any().reindex(daily.index, fill_value=False)
            hour_evaluable = evaluable.reindex(hour.index.normalize()).to_numpy()
            bad = {
                "3h without any": flag["3h"].eq(True) & ~flag["any"].eq(True),
                "any differs from its hours": evaluable & (flag["any"].eq(True) != hour_any),
                "hour flag set on a day not evaluable": pd.Series(hour.notna().to_numpy() & ~hour_evaluable),
            }
            for rule in RULES:
                start = daily[f"{source}_{d}_range_start_{b}_{rule}"]
                end = daily[f"{source}_{d}_range_end_{b}_{rule}"]
                bad[f"{rule} empty on an evaluable day or set on another"] = flag[rule].isna() == evaluable
                bad[f"{rule} range without flag or flag without range"] = flag[rule].eq(True) != (start.notna() & end.notna())
                bad[f"{rule} range with one edge"] = start.notna() != end.notna()
            problems += [f"{source} {d} {b}: {name} ({int(mask.sum())} rows)" for name, mask in bad.items() if mask.sum()]
    return problems


problems = structural_problems(DAILY_EXPORT, HOURLY_EXPORT)
if problems:
    raise RuntimeError("structural check failed:\n  " + "\n  ".join(problems))
print(f"structural checks passed on {len(FLAG_SOURCES)} sources × {len(LABEL_PAIRS)} direction-basis pairs")

# Days flagged in both directions, any basis and rule
for source in FLAG_SOURCES:
    flagged = {
        d: DAILY_EXPORT[[f"{source}_{d}_risk_{b}_{rule}" for b in BASES[d] for rule in RULES]].eq(True).any(axis=1)
        for d in BASES
    }
    print(f"days flagged in both directions, {source:<6}: {int((flagged['high'] & flagged['low']).sum())}")